In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

In [3]:
# Filter out index/unnamed columns
valid_cols = [c for c in strain_df.columns if not c.startswith('Unnamed')]

X_all = [] # Initializing the lists
y_all = []

for col in valid_cols:
    eps_raw = strain_df[col].values
    sig_raw = stress_df[col].values
    
    # NumPy Sort: Keep Strain and Stress pairs together
    idx = np.argsort(eps_raw)
    X_all.extend(eps_raw[idx])
    y_all.extend(sig_raw[idx])

# Convert lists to NumPy arrays
X = np.array(X_all)
y = np.array(y_all)

# Find the global yield point to split the data
y_idx = find_yield_index(X, y)
threshold = X[y_idx]

In [4]:
def find_yield_index(eps, sig):
    slopes = np.diff(sig) / np.diff(eps)
    # Smooth to handle experimental noise
    slopes_smoothed = np.convolve(slopes, np.ones(5)/5, mode='same')
    E_init = np.mean(slopes_smoothed[2:10])
    logic = slopes_smoothed < 0.85 * E_init
    return np.argmax(logic) if np.any(logic) else int(len(eps)*0.1)

In [19]:
import pandas as pd
import numpy as np
from scipy.optimize import curve_fit
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

# 1. LOAD DATA
strain_df = pd.read_csv("strain.csv")
stress_df = pd.read_csv("stress.csv")
valid_cols = [c for c in strain_df.columns if not c.startswith('Unnamed')]

# 2. PHYSICS DEFINITIONS
def find_yield_index(eps, sig):
    slopes = np.diff(sig) / np.diff(eps)
    slopes_sm = np.convolve(slopes, np.ones(10)/10, mode='same')
    E_init = np.mean(slopes_sm[5:15]) 
    logic = slopes_sm < 0.85 * E_init
    return np.argmax(logic) if np.any(logic) else int(len(eps) * 0.1)

# Initialize two tables: one for Strain (Inputs), one for Stress (Outputs)
test_strain_table = pd.DataFrame()
pred_stress_table = pd.DataFrame()
accuracy_metrics = []

print("--- Processing ML Predictions ---")

for col in valid_cols:
    # Prepare data
    eps = strain_df[col].dropna().values
    sig = stress_df[col].dropna().values
    
    # Clean and Sort
    mask = (eps > 0) & (sig > 0)
    eps, sig = eps[mask], sig[mask]
    idx_sort = np.argsort(eps)
    eps, sig = eps[idx_sort], sig[idx_sort]
    
    # SPLIT: 20% for testing
    X_train, X_test, y_train, y_test = train_test_split(eps, sig, test_size=0.2, random_state=42)
    
    # TRAIN: Sort training subset for physics detection
    tr_s = np.argsort(X_train)
    X_tr, y_tr = X_train[tr_s], y_train[tr_s]
    
    # Ensure y_idx is safe (fixes the ydata error)
    y_idx = find_yield_index(X_tr, y_tr)
    if y_idx < 10: y_idx = 10
    if y_idx > len(X_tr) - 10: y_idx = len(X_tr) - 10
    threshold = X_tr[y_idx]
    
    # 3. ML FITTING
    try:
        E_fit, _ = curve_fit(lambda x, E: E*x, X_tr[:y_idx], y_tr[:y_idx], p0=[200000])
        p_fit, _ = curve_fit(lambda x, K, n: K*(x**n), X_tr[y_idx:], y_tr[y_idx:], p0=[max(y_tr), 0.2])
        
        # 4. PREDICT: Apply learned physics to Test Strain (X_test)
        y_pred = np.where(X_test < threshold, 
                          E_fit[0] * X_test, 
                          p_fit[0] * (X_test**p_fit[1]))
        
        # Calculate R-squared
        r2 = r2_score(y_test, y_pred)
        accuracy_metrics.append({'Condition': col, 'R2_Score': r2})
        
        # 5. FORMATTING: Store Strain and Stress separately
        strain_col = pd.DataFrame({col: X_test}).reset_index(drop=True)
        stress_col = pd.DataFrame({col: y_pred}).reset_index(drop=True)
        
        test_strain_table = pd.concat([test_strain_table, strain_col], axis=1)
        pred_stress_table = pd.concat([pred_stress_table, stress_col], axis=1)

        
    except Exception as e:
        print(f"Error in {col}: {e}")



# 6. FINAL OUTPUT
print("\n--- TEST STRAIN VALUES (INPUTS) ---")
print(test_strain_table.head(5).to_string(index=False))

print("\n--- ML PREDICTED STRESS (OUTPUTS) ---")
print(pred_stress_table.head(5).to_string(index=False))



--- Processing ML Predictions ---

--- TEST STRAIN VALUES (INPUTS) ---
     0°C    400°C    600°C
0.008384 0.014444 0.005456
0.019664 0.005617 0.008382
0.018089 0.017917 0.015966
0.012943 0.015145 0.011610
0.008164 0.008898 0.018419

--- ML PREDICTED STRESS (OUTPUTS) ---
      0°C     400°C     600°C
57.233181 50.128262 23.055821
67.127686 42.097540 23.935959
66.087632 52.165350 25.320484
62.075974 50.569114 24.626308
56.948575 45.834178 25.638110

--- MODEL ACCURACY (R^2 VALUES) ---
Condition  R2_Score
      0°C  0.847821
    400°C  0.892305
    600°C  0.696618


In [32]:
import numpy as np

# 1. Initialize list for results
percentage_errors = []

for col in valid_cols:
    # (Assuming y_test and y_pred are available from your previous loop)
    
    # 2. Calculate the absolute percentage error for every point
    # We use np.abs and division to get the relative difference
    relative_error = np.abs((y_test - y_pred) / y_test)
    
    # 3. Take the mean and multiply by 100
    mape = np.mean(relative_error) * 100
    
    percentage_errors.append({'Condition': col, 'Error_Percent': mape})

print("\n--- MODEL ACCURACY ---\n")
accuracy_df = pd.DataFrame(accuracy_metrics)
print(accuracy_df.to_string(index=False))
print(f"\nMAPE = {mape:.2f}%")



--- MODEL ACCURACY ---

Condition  R2_Score
      0°C  0.847821
    400°C  0.892305
    600°C  0.696618

MAPE = 14.91%
